In [1]:
!pip install cupy-cuda12x torch torchvision matplotlib

In [9]:
import cupy as cp
import numpy as np
import time

def cpu_vector_add(a, b):
    return a + b

def gpu_vector_add(a, b):
    return a + b  # CuPy automatically uses GPU

sizes = [2**10, 2**14, 2**18, 2**22]

results = []

for N in sizes:
    a_cpu = np.random.rand(N).astype(np.float32)
    b_cpu = np.random.rand(N).astype(np.float32)

    # CPU timing
    start = time.time()
    cpu_vector_add(a_cpu, b_cpu)
    cpu_time = (time.time() - start) * 1000

    # GPU timing
    a_gpu = cp.asarray(a_cpu)
    b_gpu = cp.asarray(b_cpu)

    cp.cuda.Stream.null.synchronize()
    start = time.time()
    gpu_vector_add(a_gpu, b_gpu)
    cp.cuda.Stream.null.synchronize()
    gpu_time = (time.time() - start) * 1000

    results.append((N, cpu_time, gpu_time, cpu_time/gpu_time))

for r in results:
    print(f"N={r[0]}, CPU={r[1]:.2f}ms, GPU={r[2]:.2f}ms, Speedup={r[3]:.2f}")

N=1024, CPU=0.02ms, GPU=141.09ms, Speedup=0.00
N=16384, CPU=0.02ms, GPU=0.07ms, Speedup=0.32
N=262144, CPU=0.25ms, GPU=0.08ms, Speedup=3.04
N=4194304, CPU=5.57ms, GPU=0.49ms, Speedup=11.46


In [10]:
import cupy as cp
import numpy as np
import time

N = 2**20
x = cp.random.rand(N).astype(cp.float32)

# Naive
start = time.time()
res1 = cp.sum(x)
cp.cuda.Stream.null.synchronize()
t1 = time.time() - start

# Faster reduction (built-in optimized)
start = time.time()
res2 = x.sum()
cp.cuda.Stream.null.synchronize()
t2 = time.time() - start

print("Naive:", t1)
print("Optimized:", t2)
print("Match:", cp.allclose(res1, res2))

Naive: 0.023578882217407227
Optimized: 0.0002548694610595703
Match: True


In [11]:
strides = [1,2,4,8,16,32]
times = []

for s in strides:
    x = cp.random.rand(1024*1024).astype(cp.float32)
    start = time.time()
    y = x[::s].sum()
    cp.cuda.Stream.null.synchronize()
    times.append(time.time()-start)

for s,t in zip(strides,times):
    print(f"Stride {s}: {t:.6f}s")

Stride 1: 0.000649s
Stride 2: 0.115017s
Stride 4: 0.000471s
Stride 8: 0.000381s
Stride 16: 0.000286s
Stride 32: 0.000236s


In [12]:
import cupy as cp

x = cp.linspace(-4, 4, 1000000)

sigmoid = 1 / (1 + cp.exp(-x))
tanh = cp.tanh(x)
relu = cp.maximum(0, x)
leaky_relu = cp.where(x > 0, x, 0.01*x)

print("Computed all activations")

Computed all activations


In [6]:
def cross_entropy(logits, labels):
    logits = logits - cp.max(logits, axis=1, keepdims=True)
    exp = cp.exp(logits)
    softmax = exp / cp.sum(exp, axis=1, keepdims=True)

    log_probs = -cp.log(softmax[cp.arange(len(labels)), labels])
    return cp.mean(log_probs)

# test
logits = cp.random.rand(1000, 10)
labels = cp.random.randint(0, 10, size=1000)

print("Loss:", cross_entropy(logits, labels))

Loss: 2.338459285438861


In [7]:
def grad_cross_entropy(logits, labels):
    exp = cp.exp(logits - cp.max(logits, axis=1, keepdims=True))
    softmax = exp / cp.sum(exp, axis=1, keepdims=True)

    softmax[cp.arange(len(labels)), labels] -= 1
    return softmax

print("Gradient computed")

Gradient computed


In [8]:
import cupy as cp
import time

sizes = [128,256,512,1024]

for n in sizes:
    A = cp.random.rand(n,n)
    B = cp.random.rand(n,n)

    start = time.time()
    C = cp.dot(A,B)
    cp.cuda.Stream.null.synchronize()
    t = time.time() - start

    gflops = (2*n*n*n) / (t*1e9)
    print(f"N={n}, Time={t:.4f}s, GFLOPS={gflops:.2f}")

N=128, Time=0.1309s, GFLOPS=0.03
N=256, Time=0.0006s, GFLOPS=52.51
N=512, Time=0.0038s, GFLOPS=71.21
N=1024, Time=0.0290s, GFLOPS=74.10


In [9]:
import torch
import torch.nn.functional as F
import time

device = "cuda"

x = torch.randn(32,64,14,14).to(device)

# Conv2D
start = time.time()
y = F.conv2d(x, torch.randn(64,64,3,3).to(device), padding=1)
torch.cuda.synchronize()
print("Conv time:", time.time()-start)

# MaxPool
start = time.time()
y = F.max_pool2d(x,2)
torch.cuda.synchronize()
print("MaxPool time:", time.time()-start)

Conv time: 0.41498732566833496
MaxPool time: 0.011238574981689453


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

device = "cuda"

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,32,3,1)
        self.conv2 = nn.Conv2d(32,64,3,1)
        self.fc1 = nn.Linear(9216,128)
        self.fc2 = nn.Linear(128,10)

    def forward(self,x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.flatten(x,1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = CNN().to(device)

In [1]:
import torch
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(1600, 128)  # ✅ correct
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))   # 28→26
        x = self.pool(x)                # 26→13

        x = torch.relu(self.conv2(x))   # 13→11
        x = self.pool(x)                # 11→5

        x = torch.flatten(x, 1)         # 64×5×5 = 1600

        x = torch.relu(self.fc1(x))
        return self.fc2(x)

In [3]:
model = CNN().to("cuda")

In [5]:
import torch
from torchvision import datasets, transforms

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        root="./data",
        train=True,
        download=True,
        transform=transforms.ToTensor()
    ),
    batch_size=256,
    shuffle=True
)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.58MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.7MB/s]


In [7]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = torch.nn.CrossEntropyLoss()

In [8]:
scaler = torch.amp.GradScaler("cuda")

for epoch in range(2):
    for data, target in train_loader:
        data, target = data.to("cuda"), target.to("cuda")

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            output = model(data)
            loss = loss_fn(output, target)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()